# Single-Gene SNP Embedding Models

This notebook trains downstream neural models for each gene independently.
It intentionally does not compute the separate linear comparison; use `run_elasticnet_baselines.ipynb` for that.


In [ ]:
from __future__ import annotations

import json
import math
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    import pyBigWig
except ImportError as exc:
    raise ImportError("Install pyBigWig before running this notebook.") from exc


In [ ]:
PROJECT_DIR = Path("/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding")
HUMAN_DIR = Path("/mnt/rice/default/Workspace/xuxiaolong/human")
EMBEDDING_ROOT = Path("/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding_res_260708")
GENE_BED = PROJECT_DIR / "test.10gene.bed"
GENCODE_GTF = HUMAN_DIR / "gencode.v49.annotation.sorted.gtf"
SAMPLE_NAME_FILE = PROJECT_DIR / "101samples.name.txt"
BIGWIG_DIR = Path("/mnt/genos100-new/Public/CIMA/norm_CIMA_bw_101")
REFERENCE_AVERAGE_BW = PROJECT_DIR / "reference_from_train81" / "train81.average.bw"

VAL_INDIVIDUALS = ["H005", "H010", "H055", "H102", "H103", "H137", "H198", "H202", "H276", "H319"]
TEST_INDIVIDUALS = ["H030", "H117", "H118", "H129", "H195", "H197", "H215", "H225", "H261", "H309"]

RANDOM_STATE = 20260712
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

def read_sample_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", header=None, names=["accession", "sample"])
    df["accession"] = df["accession"].astype(str)
    df["sample"] = df["sample"].astype(str)
    return df

sample_table = read_sample_table(SAMPLE_NAME_FILE)
sample_to_accession = dict(zip(sample_table["sample"], sample_table["accession"]))
all_individuals = sample_table["sample"].tolist()
val_set = set(VAL_INDIVIDUALS)
test_set = set(TEST_INDIVIDUALS)
train_individuals = [x for x in all_individuals if x not in val_set and x not in test_set]

assert len(train_individuals) == len(all_individuals) - len(VAL_INDIVIDUALS) - len(TEST_INDIVIDUALS)
assert not (val_set & test_set)
assert REFERENCE_AVERAGE_BW.exists(), REFERENCE_AVERAGE_BW

def bigwig_path_for_sample(sample: str) -> Path:
    accession = sample_to_accession[sample]
    return BIGWIG_DIR / accession / "re-normalized_Monocyte.total.bw"

missing_bigwig = [sample for sample in all_individuals if not bigwig_path_for_sample(sample).exists()]
assert not missing_bigwig, f"Samples missing normalized bigWig: {missing_bigwig}"

@dataclass(frozen=True)
class GeneRecord:
    chrom: str
    start: int
    end: int
    name: str
    embedding_name: str
    strand: str
    tss: int
    transcript_id: str
    expression_regions: tuple[tuple[str, int, int], ...]

def normalize_chrom(chrom: str) -> str:
    chrom = str(chrom)
    return chrom if chrom.startswith("chr") else f"chr{chrom}"

def strip_ensembl_version(value: str) -> str:
    return str(value).split(".", 1)[0]

def transcript_id_from_name(name: str) -> str:
    for token in re.split(r"[|_]", str(name)):
        if token.startswith("ENST"):
            return token
    raise ValueError(f"Could not extract ENST transcript id from {name!r}")

def embedding_name_from_bed_name(name: str) -> str:
    parts = str(name).split("|")
    if len(parts) >= 3:
        return f"{parts[0]}_{parts[1]}_{parts[2]}"
    return str(name).replace("|", "_")

def parse_gtf_attributes(attr_text: str) -> dict[str, str]:
    return {key: value for key, value in re.findall(r'(\S+) "([^"]*)"', attr_text)}

def load_transcript_exons(gtf_path: Path, transcript_ids: Iterable[str]) -> dict[str, list[tuple[str, int, int, str]]]:
    requested = {str(tid) for tid in transcript_ids}
    requested_base = {strip_ensembl_version(tid) for tid in requested}
    exons: dict[str, list[tuple[str, int, int, str]]] = {tid: [] for tid in requested}
    with Path(gtf_path).open() as handle:
        for line in handle:
            if not line or line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) != 9 or fields[2] != "exon":
                continue
            attrs = parse_gtf_attributes(fields[8])
            transcript_id = attrs.get("transcript_id")
            if not transcript_id:
                continue
            transcript_base = strip_ensembl_version(transcript_id)
            if transcript_id in requested:
                key = transcript_id
            elif transcript_base in requested_base:
                matches = [tid for tid in requested if strip_ensembl_version(tid) == transcript_base]
                key = matches[0]
            else:
                continue
            exons.setdefault(key, []).append((normalize_chrom(fields[0]), int(fields[3]) - 1, int(fields[4]), fields[6]))
    missing = [tid for tid in requested if not exons.get(tid)]
    if missing:
        raise KeyError(f"No exon records found for transcript(s): {missing[:10]}")
    return exons

def select_three_prime_exon(exons: list[tuple[str, int, int, str]]) -> tuple[tuple[str, int, int], ...]:
    strands = {exon[3] for exon in exons}
    if len(strands) != 1:
        raise ValueError(f"Expected one strand per transcript, got {sorted(strands)}")
    strand = next(iter(strands))
    if strand == "+":
        chrom, start, end, _ = max(exons, key=lambda item: (item[2], item[1]))
    elif strand == "-":
        chrom, start, end, _ = min(exons, key=lambda item: (item[1], item[2]))
    else:
        raise ValueError(f"Unsupported strand: {strand!r}")
    return ((chrom, start, end),)

def read_test10_genes() -> list[GeneRecord]:
    raw = []
    with GENE_BED.open() as handle:
        for line in handle:
            if not line.strip() or line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            chrom, start, end, name = fields[:4]
            strand = fields[5] if len(fields) > 5 else "+"
            transcript_id = transcript_id_from_name(name)
            raw.append((normalize_chrom(chrom), int(start), int(end), name, embedding_name_from_bed_name(name), strand, transcript_id))
    exons = load_transcript_exons(GENCODE_GTF, [x[6] for x in raw])
    return [
        GeneRecord(chrom, start, end, name, embedding_name, exons[transcript_id][0][3], (start + end) // 2, transcript_id, select_three_prime_exon(exons[transcript_id]))
        for chrom, start, end, name, embedding_name, strand, transcript_id in raw
    ]

def read_all_embedding_genes() -> list[GeneRecord]:
    metas = []
    for meta_path in sorted(EMBEDDING_ROOT.glob("*/meta.json")):
        meta = json.loads(meta_path.read_text())
        transcript_id = transcript_id_from_name(meta["name"])
        metas.append((meta_path.parent.name, meta, transcript_id))
    exons = load_transcript_exons(GENCODE_GTF, [x[2] for x in metas])
    genes = []
    for embedding_name, meta, transcript_id in metas:
        start = int(meta["start"])
        end = int(meta["end"])
        chrom = normalize_chrom(meta["chrom"])
        expr = select_three_prime_exon(exons[transcript_id])
        gene_name = meta["name"].replace("_", "|", 2)
        genes.append(GeneRecord(chrom, start, end, gene_name, embedding_name, exons[transcript_id][0][3], (start + end) // 2, transcript_id, expr))
    return genes

def bw_mean_regions(bw, regions: Iterable[tuple[str, int, int]]) -> float:
    weighted_sum = 0.0
    total_bp = 0
    for chrom, start, end in regions:
        values = np.array(bw.values(chrom, max(0, int(start)), max(0, int(end)), numpy=True), dtype=np.float32)
        if values.size == 0:
            continue
        values = np.nan_to_num(values, nan=0.0)
        weighted_sum += float(values.sum())
        total_bp += int(values.size)
    if total_bp == 0:
        return float("nan")
    return weighted_sum / total_bp

def compute_targets(samples: list[str], gene: GeneRecord) -> np.ndarray:
    values = []
    with pyBigWig.open(str(REFERENCE_AVERAGE_BW)) as ref_bw:
        reference_value = bw_mean_regions(ref_bw, gene.expression_regions)
    for sample in samples:
        with pyBigWig.open(str(bigwig_path_for_sample(sample))) as sample_bw:
            sample_value = bw_mean_regions(sample_bw, gene.expression_regions)
        values.append(sample_value - reference_value)
    return np.asarray(values, dtype=np.float32)

def sample_pt_path(gene: GeneRecord, sample: str) -> Path:
    path = EMBEDDING_ROOT / gene.embedding_name / f"CIMA-{sample}_CIMA-{sample}.vcf.pt"
    if path.exists():
        return path
    matches = sorted((EMBEDDING_ROOT / gene.embedding_name).glob(f"*{sample}*.vcf.pt"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one .vcf.pt for {gene.embedding_name}/{sample}, found {len(matches)}")
    return matches[0]

def hidden_states_for_sample(gene: GeneRecord, sample: str) -> np.ndarray:
    payload = torch.load(sample_pt_path(gene, sample), map_location="cpu", weights_only=False)
    arr = payload["hidden_states"].detach().cpu().float().numpy()
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D hidden_states, got {arr.shape} for {gene.embedding_name}/{sample}")
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)

def pooled_snp_features(states: np.ndarray) -> np.ndarray:
    return np.concatenate([
        states.mean(axis=0),
        states.std(axis=0),
        states.max(axis=0),
        np.abs(states).mean(axis=0),
    ]).astype(np.float32, copy=False)

def gene_feature_matrix(gene: GeneRecord, samples: list[str], feature_mode: str) -> tuple[np.ndarray, dict[str, int]]:
    rows = []
    shapes = []
    for sample in samples:
        states = hidden_states_for_sample(gene, sample)
        shapes.append(tuple(states.shape))
        if feature_mode == "pooled":
            rows.append(pooled_snp_features(states))
        elif feature_mode == "flatten":
            rows.append(states.reshape(-1))
        else:
            raise ValueError(f"Unknown feature_mode={feature_mode!r}")
    if feature_mode == "flatten" and len(set(shapes)) != 1:
        raise ValueError(f"Flatten feature requires equal shapes for {gene.embedding_name}, got {sorted(set(shapes))[:5]}")
    X = np.vstack(rows).astype(np.float32, copy=False)
    return X, {"n_snps": int(shapes[0][0]), "embedding_dim": int(shapes[0][1]), "n_features": int(X.shape[1])}

def safe_pearson(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float("nan")
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def safe_r2(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) == 0 or np.var(y_true) == 0:
        return float("nan")
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    return float(1.0 - ss_res / ss_tot)

def regression_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "rmse": float(np.sqrt(np.mean((y_pred - y_true) ** 2))),
        "mae": float(np.mean(np.abs(y_pred - y_true))),
        "pearson": safe_pearson(y_true, y_pred),
        "r2": safe_r2(y_true, y_pred),
        "target_std": float(np.std(y_true)),
        "prediction_std": float(np.std(y_pred)),
    }

def prefixed_metrics(prefix: str, y_true, y_pred) -> dict[str, float]:
    return {f"{prefix}_{key}": value for key, value in regression_metrics(y_true, y_pred).items()}


In [ ]:
OUTPUT_DIR = PROJECT_DIR / "single_gene_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENE_SET_MODE = "test10"  # "test10" or "all97"
FEATURE_MODE = "pooled"  # "pooled" is compact; "flatten" trains a true full downstream linear/MLP head per gene.
MODEL_KIND = "linear"  # "linear" or "mlp"
BATCH_SIZE = 16
EPOCHS = 300
LR = 1e-3
WEIGHT_DECAY = 1e-3
DROPOUT = 0.2
PATIENCE = 30

genes = read_test10_genes() if GENE_SET_MODE == "test10" else read_all_embedding_genes()
print(f"device={DEVICE} genes={len(genes)} train={len(train_individuals)} val={len(VAL_INDIVIDUALS)} test={len(TEST_INDIVIDUALS)}")


In [ ]:
class ArrayDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y.reshape(-1, 1), dtype=torch.float32)

    def __len__(self):
        return int(self.X.shape[0])

    def __getitem__(self, index: int):
        return self.X[index], self.y[index]

class SingleGeneRegressor(nn.Module):
    def __init__(self, input_dim: int, model_kind: str = MODEL_KIND, dropout: float = DROPOUT):
        super().__init__()
        if model_kind == "linear":
            self.net = nn.Linear(input_dim, 1)
        elif model_kind == "mlp":
            self.net = nn.Sequential(
                nn.Linear(input_dim, 256),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(256, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, 1),
            )
        else:
            raise ValueError(f"Unknown model_kind={model_kind!r}")

    def forward(self, x):
        return self.net(x)

def standardize_from_train(X_train, X_val, X_test):
    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_val - mean) / std, (X_test - mean) / std

def train_torch_regressor(model, X_train, y_train, X_val, y_val):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()
    loader = DataLoader(ArrayDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    Xv = torch.as_tensor(X_val, dtype=torch.float32, device=DEVICE)
    yv = torch.as_tensor(y_val.reshape(-1, 1), dtype=torch.float32, device=DEVICE)
    best_state = None
    best_val = math.inf
    bad_epochs = 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        model.eval()
        with torch.no_grad():
            val_loss = float(criterion(model(Xv), yv).detach().cpu())
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_loss": val_loss})
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if bad_epochs >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

def predict_model(model, X):
    model.eval()
    with torch.no_grad():
        pred = model(torch.as_tensor(X, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy().reshape(-1)
    return pred


In [ ]:
def run_single_gene_model(gene: GeneRecord) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    samples = all_individuals
    X, info = gene_feature_matrix(gene, samples, FEATURE_MODE)
    y = compute_targets(samples, gene)
    sample_to_idx = {sample: i for i, sample in enumerate(samples)}
    train_idx = [sample_to_idx[s] for s in train_individuals]
    val_idx = [sample_to_idx[s] for s in VAL_INDIVIDUALS]
    test_idx = [sample_to_idx[s] for s in TEST_INDIVIDUALS]
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]
    X_train, X_val, X_test = standardize_from_train(X_train, X_val, X_test)

    model = SingleGeneRegressor(input_dim=X_train.shape[1])
    model, history = train_torch_regressor(model, X_train, y_train, X_val, y_val)
    pred_train = predict_model(model, X_train)
    pred_val = predict_model(model, X_val)
    pred_test = predict_model(model, X_test)

    row = {
        "gene": gene.name,
        "embedding_name": gene.embedding_name,
        "model_kind": MODEL_KIND,
        "feature_mode": FEATURE_MODE,
        **info,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "n_test": len(test_idx),
        **prefixed_metrics("train", y_train, pred_train),
        **prefixed_metrics("val", y_val, pred_val),
        **prefixed_metrics("test", y_test, pred_test),
    }
    prediction_rows = []
    for split, idxs, preds in [("train", train_idx, pred_train), ("val", val_idx, pred_val), ("test", test_idx, pred_test)]:
        for i, pred in zip(idxs, preds):
            prediction_rows.append({
                "split": split,
                "sample": samples[i],
                "gene": gene.name,
                "embedding_name": gene.embedding_name,
                "prediction_diff": float(pred),
                "target_diff": float(y[i]),
            })
    history["gene"] = gene.name
    history["embedding_name"] = gene.embedding_name
    return row, pd.DataFrame(prediction_rows), history


In [ ]:
metric_rows = []
prediction_frames = []
history_frames = []

for gene in genes:
    print(f"Training {MODEL_KIND} single-gene model: {gene.embedding_name}")
    row, pred_df, history_df = run_single_gene_model(gene)
    metric_rows.append(row)
    prediction_frames.append(pred_df)
    history_frames.append(history_df)
    print({"gene": gene.embedding_name, "test_pearson": row["test_pearson"], "test_r2": row["test_r2"], "test_rmse": row["test_rmse"]})

metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)
history = pd.concat(history_frames, ignore_index=True)

metrics_path = OUTPUT_DIR / "model_single_gene_metrics.csv"
predictions_path = OUTPUT_DIR / "model_single_gene_predictions.csv"
history_path = OUTPUT_DIR / "model_single_gene_history.csv"
metrics.to_csv(metrics_path, index=False)
predictions.to_csv(predictions_path, index=False)
history.to_csv(history_path, index=False)
print(metrics_path)
print(predictions_path)
print(history_path)
metrics.sort_values("test_pearson", ascending=False)
